In [1]:
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials

In [2]:
def getXrefs():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("/home/joe/work/client_secret.json")
    # for gg in gc.list_spreadsheet_files():
    #      print("GGGGG ",gg)
    # https://docs.google.com/spreadsheets/d/1WTaOglzbSsYiHhAGguGxHQXmAGmOhfFHkGkMLowxAOA/edit?usp=sharing
    sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'Maintenance_Framework')
    repo_sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'MetadataRepository')
    fields_sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'Field Descriptions')

    dfRepo = pd.DataFrame(repo_sheet.get_all_records(head=3))
    xrefsBy4x4 = {}
    xrefsByTitle = {}

    for index,row in dfRepo[['Dataset Title','Socrata Link']].iterrows():
        xrefsByTitle[row['Dataset Title']] = row['Socrata Link']
        xrefsBy4x4[row['Socrata Link']] = row['Dataset Title']
        
    dfFields = pd.DataFrame(fields_sheet.get_all_records(head=1))
    fields = {}
    for index,row in dfFields.iterrows():
        s4x4 = row["Socrata ID"]
        of = row["Source Field Name"]
        tf = row["Full Field Name"]
        af = row["API Field Name"]
        des = row["Description"]
        if s4x4 in fields:
            fields[s4x4]["source"].append(of)
            fields[s4x4]["cim"].append(tf)
            fields[s4x4]["api"].append(af)
            fields[s4x4]["description"].append(des)
        
        else:
            fields[s4x4] = {}
            fields[s4x4]["source"] = []
            fields[s4x4]["cim"] = []
            fields[s4x4]["api"] = []
            fields[s4x4]["description"] = []
            
            
            fields[s4x4]["source"].append(of)
            fields[s4x4]["cim"].append(tf)
            fields[s4x4]["api"].append(af)
            fields[s4x4]["description"].append(des)
            
        
        
        
    return xrefsBy4x4,xrefsByTitle,fields

In [3]:
xrefsBy4x4,xrefsByTitle,fields = getXrefs()

/tmp/ipykernel_20272/3957348916.py:1: DeprecationWarning: [Deprecated][in version 6.0.0]: client_factory will be replaced by gspread.http_client types
  xrefsBy4x4,xrefsByTitle,fields = getXrefs()


In [5]:
string = "paid"
string=string.lower()
for key,val in xrefsBy4x4.items():
    if val.lower().find(string) > -1:
        print(f"{key} - {val}\n")

ew9y-6tv9 - Paid Solicitor Solicitation Notices in Colorado

37wu-kn3g - Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado

mr4v-jz8u - Persons Associated with Charitable Organizations, Paid Solicitors, and Professional Fundraising Consultants in Colorado

wwbh-7bpa - Paid Solicitors Disclosed on Charity Registration Forms in Colorado



In [6]:
xrefsByTitle

{'Transparency Online Project (TOPS) - State Government Revenue and Expenditures in Colorado': 'rifs-n6ib',
 'Truck Station Electrification in Colorado 2014': 'c8jj-hcxj',
 'GDP by Metropolitan Statistical Area': '82s5-cpkk',
 'Personal Consumption Expenditures': 'n55r-9hud',
 'Farmers Markets in Colorado 2017': 'ms6b-y4xc',
 'Aquaculture Facilities in Colorado': 'e6e8-qmi7',
 'Degrees Awarded to Post-Secondary Graduates in Colorado': 'hxf8-ab6k',
 'Enrollment Demographics for Post-Secondary Graduates in Colorado': 'p8m4-v33g',
 'Post-Secondary Financial Aid Demographics in Colorado': 'g53r-j5td',
 'School Programs in Colorado': 'jnj7-fw37',
 'Long-Term Employment Projections in Colorado': 'gyeb-jc69',
 'Short-Term Employment Projections in Colorado': 'u2t6-bfhr',
 'Employee Counts by Industry in Colorado': 'cjkq-q9ih',
 'Unemployment Estimates in Colorado': '4e3w-qire',
 'Employment Wages in Colorado': 'busm-qa5b',
 'Personal Income in Colorado': '2cpa-vbur',
 'Population Estimates by